# Quickstart



In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from pprint import pprint

import networkx as nx
import pandas as pd

from causalchange.causal_change import CausalChange
from causalchange.config.benchmark_config import (
    SingleDataConfig,
)
from causalchange.core.types import (
    DataMode,
    GraphSearch,
    ScoreType,
    TabularContextMethod,
    TabularContextMode,
)
from experiments.benchmarks.run_methods import (
    run_sampling,
)


def print_metrics(metrics):
    for key in sorted(metrics):
        print(f"{key:32s} {metrics[key]}")


def draw_graph(graph, title=None):
    if title:
        print(title)
    nx.draw(graph, with_labels=True)

In [5]:
cfg_data = SingleDataConfig.model_validate(
    {
        "setting": "single",
        "n_nodes": 5,
        "edge_prob": 0.4,
        "n_samples": 1000,
        "nonlinearity": "tanh",
        "seed": 1,
    }
)

sample = run_sampling(cfg_data)
df = sample.df
true_g = sample.true_summary_dag


est = CausalChange(
    data_mode=DataMode.TABULAR,
    graph_search=GraphSearch.TOPIC,
    score_type=ScoreType.LIN,
)

est.fit(df)


print("Data shape:", df.shape)
print("Estimated edges:")
pprint(sorted(est.graph_.edges()))
print("\nTrue synthetic edges:")
pprint(sorted(true_g.edges()))

Data shape: (1000, 5)
Estimated edges:
[('X2', 'X0'), ('X2', 'X4'), ('X4', 'X0'), ('X4', 'X3')]

True synthetic edges:
[('X0', 'X2'), ('X4', 'X0'), ('X4', 'X3')]


In [4]:
csv_path = Path("path/to/your_data.csv")
context_col = None  # e.g. "context" for multi-context data

if csv_path.exists():
    df_user = pd.read_csv(csv_path)

    if context_col is None:
        data_mode = DataMode.TABULAR
        context_mode = TabularContextMode.SKIP
        context_method = TabularContextMethod.SKIP
    else:
        data_mode = DataMode.TAB_CONTEXTS
        context_mode = TabularContextMode.ORACLE
        context_method = TabularContextMethod.LINC

    est_user = CausalChange(
        data_mode=data_mode,
        graph_search=GraphSearch.TOPIC,
        score_type=ScoreType.GAM,
        context_mode=context_mode,
        context_method=context_method,
        context_col=context_col,
    )

    est_user.fit(df_user)

    print("Loaded:", csv_path)
    print("Data shape:", df_user.shape)
    print("Estimated edges:")
    pprint(sorted(est_user.graph_.edges()))
else:
    print(f"Edit csv_path first. File does not exist: {csv_path}")

Edit csv_path first. File does not exist: path/to/your_data.csv
